In [1]:
# libraries imports
import pandas as pd
import numpy as np
import scipy
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from matplotlib.ticker import FormatStrFormatter, FuncFormatter
from utils import *

import glob
import time
import datetime

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


In [2]:
df_MAV = pd.read_csv('../../outputs/files/models/MAV/best_model.csv', sep=";")
df_MAV.dateStart = pd.to_datetime(df_MAV.dateStart)

df_SEV = pd.read_csv('../../outputs/files/models/SEV/best_model.csv', sep=";")
df_SEV.dateStart = pd.to_datetime(df_SEV.dateStart)

df_SEC = pd.read_csv('../../outputs/files/models/CLICHY/best_model.csv', sep=";")
df_SEC.dateStart = pd.to_datetime(df_SEC.dateStart)

ground_truth_file_MAV = pd.read_csv('../../outputs/files/census_pop/MAV_GT_manuscript.csv', sep=";")
ground_truth_file_MAV.dateStart = pd.to_datetime(ground_truth_file_MAV.dateStart)
ground_truth_file_MAV.rename(columns={'final_population':'Nt'}, inplace=True)

ground_truth_file_SEV = pd.read_csv('../../outputs/files/census_pop/SEV_GT_manuscript.csv', sep=";")
ground_truth_file_SEV.dateStart = pd.to_datetime(ground_truth_file_SEV.dateStart)
ground_truth_file_SEV.rename(columns={'final_population':'Nt'}, inplace=True)

df_been_MAV = pd.read_csv('../../outputs/files/models/MAV/model_been_original.csv', sep=";")
df_been_MAV.dateStart = pd.to_datetime(df_been_MAV.dateStart)

df_vn_MAV = pd.read_csv('../../outputs/files/models/MAV/model_vn_original.csv', sep=";")
df_vn_MAV.dateStart = pd.to_datetime(df_vn_MAV.dateStart)

df_zheng_MAV = pd.read_csv('../../outputs/files/models/MAV/model_zheng_original.csv', sep=";")
df_zheng_MAV.dateStart = pd.to_datetime(df_zheng_MAV.dateStart)

df_been_MAV.loc[df_been_MAV.dateStart.isin(ground_truth_file_MAV.dateStart), 'GT_pop'] = ground_truth_file_MAV.Nt.values
df_vn_MAV.loc[df_vn_MAV.dateStart.isin(ground_truth_file_MAV.dateStart), 'GT_pop'] = ground_truth_file_MAV.Nt.values
df_zheng_MAV.loc[df_zheng_MAV.dateStart.isin(ground_truth_file_MAV.dateStart), 'GT_pop'] = ground_truth_file_MAV.Nt.values

df_been_SEV = pd.read_csv('../../outputs/files/models/SEV/model_been_original.csv', sep=";")
df_been_SEV.dateStart = pd.to_datetime(df_been_SEV.dateStart)

df_vn_SEV = pd.read_csv('../../outputs/files/models/SEV/model_vn_original.csv', sep=";")
df_vn_SEV.dateStart = pd.to_datetime(df_vn_SEV.dateStart)

df_zheng_SEV = pd.read_csv('../../outputs/files/models/SEV/model_zheng_original.csv', sep=";")
df_zheng_SEV.dateStart = pd.to_datetime(df_zheng_SEV.dateStart)

df_been_SEV.loc[df_been_SEV.dateStart.isin(ground_truth_file_SEV.dateStart), 'GT_pop'] = ground_truth_file_SEV.Nt.values
df_vn_SEV.loc[df_vn_SEV.dateStart.isin(ground_truth_file_SEV.dateStart), 'GT_pop'] = ground_truth_file_SEV.Nt.values
df_zheng_SEV.loc[df_zheng_SEV.dateStart.isin(ground_truth_file_SEV.dateStart), 'GT_pop'] = ground_truth_file_SEV.Nt.values

df_MAV.loc[df_MAV.dateStart.isin(ground_truth_file_MAV.dateStart), 'GT_pop'] = ground_truth_file_MAV.Nt.values
df_SEV.loc[df_SEV.dateStart.isin(ground_truth_file_SEV.dateStart), 'GT_pop'] = ground_truth_file_SEV.Nt.values

df_MAV.loc[~df_MAV['GT_pop'].isna()]

,dateStart,plantVolume,DBO,DCO,MES,NGL,NH4,NTK,PT,Pluviométrie,month,year,day,Nt_hat,Nt_hat_CIL,Nt_hat_CIU,GT_pop
0,2020-01-01,57316.0,299.0,681,257.0,73.50,64.907,72.60,6.54,0.104712,1.0,2020.0,1.0,330304.114451,253578.038961,360997.684545,332191.0
1,2020-01-02,59753.0,286.0,615,253.0,66.60,59.588,65.70,7.05,0.142115,1.0,2020.0,2.0,300644.054267,291995.186722,344252.180222,332191.0
2,2020-01-03,60132.0,281.0,616,252.0,64.20,59.202,63.30,6.63,0.512019,1.0,2020.0,3.0,297340.080174,257278.214283,334897.221042,332191.0
3,2020-01-04,58092.0,291.0,655,280.0,66.70,62.988,65.80,6.93,0.000000,1.0,2020.0,4.0,308191.237347,284086.014821,334709.191998,332191.0
4,2020-01-05,58520.0,305.0,623,267.0,66.30,64.005,65.40,7.13,0.000000,1.0,2020.0,5.0,310223.654368,260443.423788,336991.892544,332191.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2020-12-27,80975.0,280.0,566,252.0,49.60,39.511,48.70,5.08,11.300000,12.0,2020.0,362.0,316610.832386,283201.188064,374928.271810,332191.0
362,2020-12-28,85734.0,233.0,430,284.0,43.95,32.093,41.70,4.62,11.400000,12.0,2020.0,363.0,306043.070543,282033.527543,360211.173009,332191.0
363,2020-12-29,78798.0,221.0,491,272.0,50.88,45.061,48.70,5.35,2.700000,12.0,2020.0,364.0,316384.951219,274867.872290,346509.701184,332191.0
364,2020-12-30,79602.0,200.0,518,211.0,55.03,46.529,53.50,5.26,1.900000,12.0,2020.0,365.0,324105.315115,312391.237523,368358.138712,332191.0


In [3]:
y_ww_mav = df_MAV.loc[~df_MAV.GT_pop.isna()].Nt_hat.values
y_insee_mav = df_MAV.loc[~df_MAV.GT_pop.isna()].GT_pop.values

y_ww_sev = df_SEV.loc[~df_SEV.GT_pop.isna()].Nt_hat.values
y_insee_sev = df_SEV.loc[~df_SEV.GT_pop.isna()].GT_pop.values

y_been_mav = df_been_MAV.loc[~df_been_MAV.GT_pop.isna()].Nt_hat.values
y_vn_mav = df_vn_MAV.loc[~df_vn_MAV.GT_pop.isna()].Nt_hat.values
y_zheng_mav = df_zheng_MAV.loc[~df_zheng_MAV.GT_pop.isna()].Nt_hat.values

y_been_sev = df_been_SEV.loc[~df_been_SEV.GT_pop.isna()].Nt_hat.values
y_vn_sev = df_vn_SEV.loc[~df_vn_SEV.GT_pop.isna()].Nt_hat.values
y_zheng_sev = df_zheng_SEV.loc[~df_zheng_SEV.GT_pop.isna()].Nt_hat.values

print("========")
print("MAV")
print("========")
print(f'MAPE MAV our model:        {mean_absolute_percentage_error(y_insee_mav, y_ww_mav)}.')
print(f'MAPE MAV Been et al.:      {mean_absolute_percentage_error(y_insee_mav, y_been_mav)}.')
print(f'MAPE MAV Van Nuijs et al.: {mean_absolute_percentage_error(y_insee_mav, y_vn_mav)}.')
print(f'MAPE MAV Zheng et al.:     {mean_absolute_percentage_error(y_insee_mav, y_zheng_mav)}.')
print("========")
print("SEV")
print("========")
print(f'MAPE SEV our model:        {mean_absolute_percentage_error(y_insee_sev, y_ww_sev)}.')
print(f'MAPE SEV Been et al.:      {mean_absolute_percentage_error(y_insee_sev, y_been_sev)}.')
print(f'MAPE SEV Van Nuijs et al.: {mean_absolute_percentage_error(y_insee_sev, y_vn_sev)}.')
print(f'MAPE SEV Zheng et al.:     {mean_absolute_percentage_error(y_insee_sev, y_zheng_sev)}.')

MAV
MAPE MAV our model:        0.06151625039169468.
MAPE MAV Been et al.:      0.12177966987016561.
MAPE MAV Van Nuijs et al.: 0.08742053876376558.
MAPE MAV Zheng et al.:     0.11844864807687666.
SEV
MAPE SEV our model:        0.06490939364188075.
MAPE SEV Been et al.:      0.17751603460077456.
MAPE SEV Van Nuijs et al.: 0.09673095078605967.
MAPE SEV Zheng et al.:     0.16480389796239925.


In [4]:
ape_mav = ape(y_insee_mav, y_ww_mav)
ape_sev = ape(y_insee_sev, y_ww_sev)

ape_mav_been = ape(y_insee_mav, y_been_mav)
ape_sev_been = ape(y_insee_sev, y_been_sev)

ape_mav_vn = ape(y_insee_mav, y_vn_mav)
ape_sev_vn = ape(y_insee_sev, y_vn_sev)

ape_mav_zheng = ape(y_insee_mav, y_zheng_mav)
ape_sev_zheng = ape(y_insee_sev, y_zheng_sev)

In [5]:
flierprops = dict(marker='o', markerfacecolor='dimgray', markersize=12, markeredgecolor='black')
whiskerprops = dict(linestyle='-',linewidth=3, color='black')
capprops = dict(linestyle='-',linewidth=3, color='black')
medianprops = dict(linewidth=2.5, color='black')
whiskers = (2.5,97.5)
patch_artist = True
showfliers=True
width = np.array([.15])
box_labels = ['MA', 'SV']
y_label = 'Absolute percentage error (\%)'
digit_y_label = FuncFormatter(format_func_0f)

# Background and grid styles :
background_facecolor = '0.9'
grid_color = 'black'
grid_linewidth = 1.5
grid_linestyle = '-'

params_dict = {}
params_dict['whiskers'] = whiskers
params_dict['patch_artist'] = patch_artist
params_dict['medianprops'] = medianprops
params_dict['flierprops'] = flierprops
params_dict['whiskerprops'] = whiskerprops
params_dict['capprops'] = capprops
params_dict['showfliers'] = showfliers
params_dict['width'] = width

In [6]:
# lockdowns only
fl_sd, fl_ed = pd.to_datetime('2020-03-17'), pd.to_datetime('2020-05-10')
sl_sd, sl_ed = pd.to_datetime('2020-10-30'), pd.to_datetime('2020-12-14')

In [7]:
symbols_dict = {}
symbols_dict['Our model | MAV-SEV'] = append_symbols_dict(ape_sev, ape_mav)
symbols_dict['Been | MAV-SEV'] = append_symbols_dict(ape_sev_been, ape_mav_been)
symbols_dict['VN | MAV-SEV'] = append_symbols_dict(ape_sev_vn, ape_mav_vn)
symbols_dict['Zheng | MAV-SEV'] = append_symbols_dict(ape_sev_zheng, ape_mav_zheng)

symbols_dict['MAV | Our model vs Been'] = append_symbols_dict(ape_mav, ape_mav_been)
symbols_dict['MAV | Our model vs VN'] = append_symbols_dict(ape_mav, ape_mav_vn)
symbols_dict['MAV | Our model vs Zheng'] = append_symbols_dict(ape_mav, ape_mav_zheng)

symbols_dict['SEV | Our model vs Been'] = append_symbols_dict(ape_sev, ape_sev_been)
symbols_dict['SEV | Our model vs VN'] = append_symbols_dict(ape_sev, ape_sev_vn)
symbols_dict['SEV | Our model vs Zheng'] = append_symbols_dict(ape_sev, ape_sev_zheng)

p-value:0.370, symbol:n.s., Cohen's d:0.066
p-value:0.000, symbol:***, Cohen's d:0.561
p-value:0.162, symbol:n.s., Cohen's d:0.104
p-value:0.000, symbol:***, Cohen's d:0.470
p-value:0.000, symbol:***, Cohen's d:0.711
p-value:0.000, symbol:***, Cohen's d:0.329
p-value:0.000, symbol:***, Cohen's d:0.672
p-value:0.000, symbol:***, Cohen's d:1.545
p-value:0.000, symbol:***, Cohen's d:0.475
p-value:0.000, symbol:***, Cohen's d:1.392


In [ ]:
smoothed_width = 3
black_width = 1
smoothed_width_biblio = 1

with plt.style.context(['science', 'notebook', 'grid']):

    KEY_SIZE = 48
    LABEL_SIZE = 40
    TICK_SIZE = 40
    TITLE_SIZE = 46
    LEGEND_SIZE = 36
    DATES_SIZE = 18
    figsize = (32, 20) #figsize = (32, 10)
    
    plt.rc('axes', labelsize=LABEL_SIZE)
    plt.rc('xtick', labelsize=TICK_SIZE)   
    plt.rc('ytick', labelsize=TICK_SIZE)
    plt.rc('figure', titlesize=TITLE_SIZE)
    plt.rc('legend', fontsize=LEGEND_SIZE)
    plt.rcParams['text.usetex'] = True
    
    fig = plt.figure(figsize=figsize, layout="constrained")
    
    ax_dict = fig.subplot_mosaic(
        """
        AABBC
        DDDDD
        """,
        gridspec_kw={'wspace': 0.05, 'hspace':0.075}
    )
    ################################ ------------------------------ A ------------------------------ ################################
    GT_mav = ax_dict['A'].plot(df_MAV.dateStart.values, df_MAV.GT_pop.values, label='Population estimated from census data', color='red', linewidth=smoothed_width, zorder=3)
    ax_dict['A'].plot(df_MAV.dateStart.values, df_MAV.GT_pop.values, color='black', linewidth=black_width, zorder=3)
    
    ax_dict['A'].plot(df_MAV.dateStart.values, 1.1*df_MAV.GT_pop.values, linestyle='--', color='red', label='90% Census data interval', linewidth=smoothed_width, alpha=0.3)
    ax_dict['A'].plot(df_MAV.dateStart.values, 0.9*df_MAV.GT_pop.values, linestyle='--', color='red', linewidth=smoothed_width, alpha=0.3)
        
    ax_dict['A'].plot(df_vn_MAV.dateStart.values, df_vn_MAV.Nt_hat.values, label='van Nuijs et al., 2011', color='forestgreen', linewidth=smoothed_width_biblio)
    ax_dict['A'].plot(df_zheng_MAV.dateStart.values, df_zheng_MAV.Nt_hat.values, label='Zheng et al., 2019', color='darkorchid', linewidth=smoothed_width_biblio)
    ax_dict['A'].plot(df_been_MAV.dateStart.values, df_been_MAV.Nt_hat.values, label='Been et al., 2014', color='dodgerblue', linewidth=smoothed_width_biblio)
    
    smoothed_1 = ax_dict['A'].plot(df_MAV.dateStart.values, df_MAV.Nt_hat.values, label='The proposed model', color='orange', linewidth=smoothed_width)
    ax_dict['A'].plot(df_MAV.dateStart.values, df_MAV.Nt_hat.values, color='black', linewidth=black_width)
    
    CIS_1 = ax_dict['A'].fill_between(df_MAV.dateStart.values, 
                        df_MAV.Nt_hat_CIL.values,
                        df_MAV.Nt_hat_CIU.values, alpha=.2, color='orange', label='95% CI')


    ax_dict['A'].axvspan(fl_sd, fl_ed, alpha=0.2, color='silver', label='Lockdowns')
    ax_dict['A'].axvspan(sl_sd, sl_ed, alpha=0.2, color='silver')
    
    ax_dict['A'].set_title('Marne Aval', size=TITLE_SIZE)
    ax_dict['A'].set_ylabel("$N_t$")
    ax_dict['A'].set_xlabel("Sampling date")
    ax_dict['A'].tick_params(axis='x', labelsize=TICK_SIZE, rotation=45)
    ax_dict['A'].tick_params(axis='y', labelsize=TICK_SIZE)
    ax_dict['A'].ticklabel_format(style='sci', axis='y', scilimits=(6,6))

    ################################ ------------------------------ B ------------------------------ ################################
    GT_sev = ax_dict['B'].plot(df_SEV.dateStart.values, df_SEV.GT_pop.values, label='Population estimated from census data', color='red', linewidth=smoothed_width, zorder=3)
    ax_dict['B'].plot(df_SEV.dateStart.values, df_SEV.GT_pop.values, color='black', linewidth=black_width, zorder=3)
    
    ax_dict['B'].plot(df_SEV.dateStart.values, 1.1*df_SEV.GT_pop.values, linestyle='--', color='red', linewidth=smoothed_width, alpha=0.3)
    ax_dict['B'].plot(df_SEV.dateStart.values, 0.9*df_SEV.GT_pop.values, linestyle='--', color='red', linewidth=smoothed_width, alpha=0.3)
        
    ax_dict['B'].plot(df_vn_SEV.dateStart.values, df_vn_SEV.Nt_hat.values, label='van Nuijs et al., 2011', color='forestgreen', linewidth=smoothed_width_biblio)
    ax_dict['B'].plot(df_zheng_SEV.dateStart.values, df_zheng_SEV.Nt_hat.values, label='Zheng et al., 2019', color='darkorchid', linewidth=smoothed_width_biblio)
    ax_dict['B'].plot(df_been_SEV.dateStart.values, df_been_SEV.Nt_hat.values, label='Been et al., 2024', color='dodgerblue', linewidth=smoothed_width_biblio)
    
    smoothed_2 = ax_dict['B'].plot(df_SEV.dateStart.values, df_SEV.Nt_hat.values, label='The proposed model', color='orange', linewidth=smoothed_width)
    ax_dict['B'].plot(df_SEV.dateStart.values, df_SEV.Nt_hat.values, color='black', linewidth=black_width)
    
    CIS_2 = ax_dict['B'].fill_between(df_SEV.dateStart.values, 
                        df_SEV.Nt_hat_CIL.values,
                        df_SEV.Nt_hat_CIU.values, alpha=.2, color='orange', label='95% CI')

    ax_dict['B'].axvspan(fl_sd, fl_ed, alpha=0.2, color='silver')
    ax_dict['B'].axvspan(sl_sd, sl_ed, alpha=0.2, color='silver')

    ax_dict['B'].set_title('Seine Valenton', size=TITLE_SIZE)
    ax_dict['B'].set_ylabel("$N_t$")
    ax_dict['B'].set_xlabel("Sampling date")
    ax_dict['B'].tick_params(axis='x', labelsize=TICK_SIZE, rotation=45)
    ax_dict['B'].tick_params(axis='y', labelsize=TICK_SIZE)

    ################################ ------------------------------ C ------------------------------ ################################    
    plot_boxplot(ax_dict, 100*ape_mav, 'C', color='orange', hatch='/', position=0.1, legend='Marne Aval', params_dict=params_dict)
    plot_boxplot(ax_dict, 100*ape_mav_zheng, 'C', color='darkorchid', hatch='/', position=-0.1, legend='Marne Aval', params_dict=params_dict)
    plot_boxplot(ax_dict, 100*ape_mav_been, 'C', color='dodgerblue', hatch='/', position=-0.3, legend='Marne Aval', params_dict=params_dict)
    plot_boxplot(ax_dict, 100*ape_mav_vn, 'C', color='forestgreen', hatch='/', position=0.3, legend='Marne Aval', params_dict=params_dict)

    plot_boxplot(ax_dict, 100*ape_sev, 'C', color='orange', hatch='X', position=1+0.1, legend='Seine Valenton', params_dict=params_dict)
    plot_boxplot(ax_dict, 100*ape_sev_zheng, 'C', color='darkorchid', hatch='X', position=1-0.1, legend='Seine Valenton', params_dict=params_dict)
    plot_boxplot(ax_dict, 100*ape_sev_been, 'C', color='dodgerblue', hatch='/', position=1-0.3, legend='Seine Valenton', params_dict=params_dict)
    plot_boxplot(ax_dict, 100*ape_sev_vn, 'C', color='forestgreen', hatch='/', position=1+0.3, legend='Seine Valenton', params_dict=params_dict)

    ### Statistical significance
    j_val = (1.01 - 0.7) / 0.05

    plot_stat_signif(ax_dict, 'C', symbols_dict['MAV | Our model vs VN'], 0.1, 0.3, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['MAV | Our model vs Zheng'], 0.1, -0.1, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['MAV | Our model vs Been'], -0.3, 0.1, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    
    plot_stat_signif(ax_dict, 'C', symbols_dict['SEV | Our model vs VN'], 1+0.1, 1+0.3, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['SEV | Our model vs Zheng'], 1+0.1, 1-0.1, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['SEV | Our model vs Been'], 1-0.3, 1+0.1, LABEL_SIZE, bar_spacing=0.004, j=j_val)

    plot_stat_signif(ax_dict, 'C', symbols_dict['VN | MAV-SEV'], 0.3, 1+0.3, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['Our model | MAV-SEV'], 0.1, 1+0.1, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['Zheng | MAV-SEV'], -0.1, 1-0.1, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    plot_stat_signif(ax_dict, 'C', symbols_dict['Been | MAV-SEV'], -0.3, 1-0.3, LABEL_SIZE, bar_spacing=0.004, j=j_val)
    
    # Specifying xlabels:
    plt.rcParams['text.usetex'] = True
    ax_dict['C'].set_xticks(np.arange(len(box_labels)))
    ax_dict['C'].set_xticklabels(box_labels, size=LABEL_SIZE)

    # Specifying ylabels:
    ax_dict['C'].set_ylabel(y_label)

    # Background and grid formatting:
    #ax_dict['C'].set_facecolor(background_facecolor)
    #ax_dict['C'].grid(color=grid_color,
    #                 linewidth=grid_linewidth,
    #                 linestyle=grid_linestyle)

    # Specifying y axis float definition:
    ax_dict['C'].yaxis.set_major_formatter(digit_y_label)

    ################################ ------------------------------ D ------------------------------ ################################ 
    smoothed_3 = ax_dict['D'].plot(df_SEC.dateStart.values, df_SEC.Nt_hat.values, label='The proposed model', color='orange', linewidth=smoothed_width)
    ax_dict['D'].plot(df_SEC.dateStart.values, df_SEC.Nt_hat.values, color='black', linewidth=black_width)
    
    CIS_3 = ax_dict['D'].fill_between(df_SEC.dateStart.values, 
                        df_SEC.Nt_hat_CIL.values,
                        df_SEC.Nt_hat_CIU.values, alpha=.2, color='orange', label='95% CI')


    ax_dict['D'].set_title('Clichy', size=TITLE_SIZE)
    ax_dict['D'].set_ylabel("$N_t$")
    ax_dict['D'].set_xlabel("Sampling date")
    ax_dict['D'].tick_params(axis='x', labelsize=TICK_SIZE, rotation=45)
    ax_dict['D'].tick_params(axis='y', labelsize=TICK_SIZE)


    # Display subplot keys
    plt.rcParams['text.usetex'] = False
    fig.canvas.draw()

    # Function to align text with the ylabel of a specific axis
    def align_text_with_ylabel(ax, text, fig):
        ylabel = ax.yaxis.label
        bbox = ylabel.get_window_extent()
        bbox_fig = fig.transFigure.inverted().transform(bbox)
        ylabel_center_fig_x = (bbox_fig[0, 0] + bbox_fig[1, 0]) / 2
        ylabel_center_fig_y = (bbox_fig[0, 1] + bbox_fig[1, 1]) / 2
        fig.text(ylabel_center_fig_x, ylabel_center_fig_y + 0.225, text, ha='center', va='center', size=KEY_SIZE, weight='bold')

    # Align text with the ylabels for each subplot
    for n, (key, ax) in enumerate(ax_dict.items()):
        align_text_with_ylabel(ax, key, fig)
    
    plt.rcParams['text.usetex'] = False
    h1, l1 = ax_dict['A'].get_legend_handles_labels()
    fig.legend(h1, l1, loc='upper center', bbox_to_anchor=(0.5, 0), fancybox=True, shadow=True, ncol=4)
    plt.savefig('../../outputs/figs/2025-12-01_MAV_SEV_Pop_Estimation.pdf', bbox_inches = 'tight')


In [9]:
# error before first lockdown
y_insee_mav_1 = df_MAV.loc[df_MAV.dateStart<fl_sd].GT_pop.values
y_insee_sev_1 = df_SEV.loc[df_SEV.dateStart<fl_sd].GT_pop.values

y_ww_mav_1 = df_MAV.loc[df_MAV.dateStart<fl_sd].Nt_hat.values
y_ww_sev_1 = df_SEV.loc[df_SEV.dateStart<fl_sd].Nt_hat.values
ape_mav_1 = ape(y_insee_mav_1, y_ww_mav_1)
ape_sev_1 = ape(y_insee_sev_1, y_ww_sev_1)

y_been_mav_1 = df_been_MAV.loc[df_been_MAV.dateStart<fl_sd].Nt_hat.values
y_been_sev_1 = df_been_SEV.loc[df_been_SEV.dateStart<fl_sd].Nt_hat.values
ape_mav_been_1 = ape(y_insee_mav_1, y_been_mav_1)
ape_sev_been_1 = ape(y_insee_sev_1, y_been_sev_1)

y_zheng_mav_1 = df_zheng_MAV.loc[df_zheng_MAV.dateStart<fl_sd].Nt_hat.values
y_zheng_sev_1 = df_zheng_SEV.loc[df_zheng_SEV.dateStart<fl_sd].Nt_hat.values
ape_mav_zheng_1 = ape(y_insee_mav_1, y_zheng_mav_1)
ape_sev_zheng_1 = ape(y_insee_sev_1, y_zheng_sev_1)

y_vn_mav_1 = df_vn_MAV.loc[df_vn_MAV.dateStart<fl_sd].Nt_hat.values
y_vn_sev_1 = df_vn_SEV.loc[df_vn_SEV.dateStart<fl_sd].Nt_hat.values
ape_mav_vn_1 = ape(y_insee_mav_1, y_vn_mav_1)
ape_sev_vn_1 = ape(y_insee_sev_1, y_vn_sev_1)

In [10]:
# error during lockdowns
sub_df_MAV = df_MAV.loc[( (df_MAV.dateStart>=fl_sd) & (df_MAV.dateStart<=fl_ed) )|( (df_MAV.dateStart>=sl_sd) & (df_MAV.dateStart<=sl_ed) )]
sub_df_SEV = df_SEV.loc[( (df_SEV.dateStart>=fl_sd) & (df_SEV.dateStart<=fl_ed) )|( (df_SEV.dateStart>=sl_sd) & (df_SEV.dateStart<=sl_ed) )]
y_insee_mav_2 = sub_df_MAV.GT_pop.values
y_insee_sev_2 = sub_df_SEV.GT_pop.values

y_ww_mav_2 = sub_df_MAV.Nt_hat.values
y_ww_sev_2 = sub_df_SEV.Nt_hat.values
ape_mav_2 = ape(y_insee_mav_2, y_ww_mav_2)
ape_sev_2 = ape(y_insee_sev_2, y_ww_sev_2)

sub_df_MAV = df_zheng_MAV.loc[( (df_zheng_MAV.dateStart>=fl_sd) & (df_zheng_MAV.dateStart<=fl_ed) )|( (df_zheng_MAV.dateStart>=sl_sd) & (df_zheng_MAV.dateStart<=sl_ed) )]
sub_df_SEV = df_zheng_SEV.loc[( (df_zheng_SEV.dateStart>=fl_sd) & (df_zheng_SEV.dateStart<=fl_ed) )|( (df_zheng_SEV.dateStart>=sl_sd) & (df_zheng_SEV.dateStart<=sl_ed) )]
y_zheng_mav_2 = sub_df_MAV.Nt_hat.values
y_zheng_sev_2 = sub_df_SEV.Nt_hat.values
ape_mav_zheng_2 = ape(y_insee_mav_2, y_zheng_mav_2)
ape_sev_zheng_2 = ape(y_insee_sev_2, y_zheng_sev_2)

sub_df_MAV = df_vn_MAV.loc[( (df_vn_MAV.dateStart>=fl_sd) & (df_vn_MAV.dateStart<=fl_ed) )|( (df_vn_MAV.dateStart>=sl_sd) & (df_vn_MAV.dateStart<=sl_ed) )]
sub_df_SEV = df_vn_SEV.loc[( (df_vn_SEV.dateStart>=fl_sd) & (df_vn_SEV.dateStart<=fl_ed) )|( (df_vn_SEV.dateStart>=sl_sd) & (df_vn_SEV.dateStart<=sl_ed) )]
y_vn_mav_2 = sub_df_MAV.Nt_hat.values
y_vn_sev_2 = sub_df_SEV.Nt_hat.values
ape_mav_vn_2 = ape(y_insee_mav_2, y_vn_mav_2)
ape_sev_vn_2 = ape(y_insee_sev_2, y_vn_sev_2)

sub_df_MAV = df_been_MAV.loc[( (df_been_MAV.dateStart>=fl_sd) & (df_been_MAV.dateStart<=fl_ed) )|( (df_been_MAV.dateStart>=sl_sd) & (df_been_MAV.dateStart<=sl_ed) )]
sub_df_SEV = df_been_SEV.loc[( (df_been_SEV.dateStart>=fl_sd) & (df_been_SEV.dateStart<=fl_ed) )|( (df_been_SEV.dateStart>=sl_sd) & (df_been_SEV.dateStart<=sl_ed) )]
y_been_mav_2 = sub_df_MAV.Nt_hat.values
y_been_sev_2 = sub_df_SEV.Nt_hat.values
ape_mav_been_2 = ape(y_insee_mav_2, y_been_mav_2)
ape_sev_been_2 = ape(y_insee_sev_2, y_been_sev_2)

In [11]:
# error between lockdowns
sub_df_MAV = df_MAV.loc[( (df_MAV.dateStart>fl_ed) & (df_MAV.dateStart<sl_sd) )]
sub_df_SEV = df_SEV.loc[( (df_SEV.dateStart>fl_ed) & (df_SEV.dateStart<sl_sd) )]
y_insee_mav_3 = sub_df_MAV.GT_pop.values
y_insee_sev_3 = sub_df_SEV.GT_pop.values

y_ww_mav_3 = sub_df_MAV.Nt_hat.values
y_ww_sev_3 = sub_df_SEV.Nt_hat.values
ape_mav_3 = ape(y_insee_mav_3, y_ww_mav_3)
ape_sev_3 = ape(y_insee_sev_3, y_ww_sev_3)

sub_df_MAV = df_zheng_MAV.loc[( (df_zheng_MAV.dateStart>fl_ed) & (df_zheng_MAV.dateStart<sl_sd) )]
sub_df_SEV = df_zheng_SEV.loc[( (df_zheng_SEV.dateStart>fl_ed) & (df_zheng_SEV.dateStart<sl_sd) )]
y_zheng_mav_3 = sub_df_MAV.Nt_hat.values
y_zheng_sev_3 = sub_df_SEV.Nt_hat.values
ape_mav_zheng_3 = ape(y_insee_mav_3, y_zheng_mav_3)
ape_sev_zheng_3 = ape(y_insee_sev_3, y_zheng_sev_3)

sub_df_MAV = df_vn_MAV.loc[( (df_vn_MAV.dateStart>fl_ed) & (df_vn_MAV.dateStart<sl_sd) )]
sub_df_SEV = df_vn_SEV.loc[( (df_vn_SEV.dateStart>fl_ed) & (df_vn_SEV.dateStart<sl_sd) )]
y_vn_mav_3 = sub_df_MAV.Nt_hat.values
y_vn_sev_3 = sub_df_SEV.Nt_hat.values
ape_mav_vn_3 = ape(y_insee_mav_3, y_vn_mav_3)
ape_sev_vn_3 = ape(y_insee_sev_3, y_vn_sev_3)

sub_df_MAV = df_been_MAV.loc[( (df_been_MAV.dateStart>fl_ed) & (df_been_MAV.dateStart<sl_sd) )]
sub_df_SEV = df_been_SEV.loc[( (df_been_SEV.dateStart>fl_ed) & (df_been_SEV.dateStart<sl_sd) )]
y_been_mav_3 = sub_df_MAV.Nt_hat.values
y_been_sev_3 = sub_df_SEV.Nt_hat.values
ape_mav_been_3 = ape(y_insee_mav_3, y_been_mav_3)
ape_sev_been_3 = ape(y_insee_sev_3, y_been_sev_3)

In [12]:
symbols_dict_A = {}
symbols_dict_A['MAV - During vs before'] = append_symbols_dict(ape_mav_2, ape_mav_1)
symbols_dict_A['MAV - During vs whole'] = append_symbols_dict(ape_mav_2, ape_mav)
symbols_dict_A['MAV - During vs between'] = append_symbols_dict(ape_mav_2, ape_mav_3)
symbols_dict_A['MAV - Before vs between'] = append_symbols_dict(ape_mav_1, ape_mav_3)
symbols_dict_A['MAV - Whole vs between'] = append_symbols_dict(ape_mav, ape_mav_3)
symbols_dict_A['MAV - Whole vs before'] = append_symbols_dict(ape_mav, ape_mav_1)

symbols_dict_A['SEV - During vs before'] = append_symbols_dict(ape_sev_2, ape_sev_1)
symbols_dict_A['SEV - During vs whole'] = append_symbols_dict(ape_sev_2, ape_sev)
symbols_dict_A['SEV - During vs between'] = append_symbols_dict(ape_sev_2, ape_sev_3)
symbols_dict_A['SEV - Before vs between'] = append_symbols_dict(ape_sev_1, ape_sev_3)
symbols_dict_A['SEV - Whole vs between'] = append_symbols_dict(ape_sev, ape_sev_3)
symbols_dict_A['SEV - Whole vs before'] = append_symbols_dict(ape_sev, ape_sev_1)

p-value:0.066, symbol:n.s., Cohen's d:0.280
p-value:0.051, symbol:n.s., Cohen's d:0.210
p-value:0.034, symbol:*, Cohen's d:0.258
p-value:0.784, symbol:n.s., Cohen's d:0.035
p-value:0.542, symbol:n.s., Cohen's d:0.058
p-value:0.781, symbol:n.s., Cohen's d:0.032
p-value:0.000, symbol:***, Cohen's d:0.878
p-value:0.000, symbol:***, Cohen's d:0.438
p-value:0.000, symbol:***, Cohen's d:0.608
p-value:0.037, symbol:*, Cohen's d:0.287
p-value:0.082, symbol:n.s., Cohen's d:0.159
p-value:0.000, symbol:***, Cohen's d:0.434


In [13]:
symbols_dict_B = {}
symbols_dict_B['MAV - During vs before'] = append_symbols_dict(ape_mav_vn_2, ape_mav_vn_1)
symbols_dict_B['MAV - During vs whole'] = append_symbols_dict(ape_mav_vn_2, ape_mav_vn)
symbols_dict_B['MAV - During vs between'] = append_symbols_dict(ape_mav_vn_2, ape_mav_vn_3)
symbols_dict_B['MAV - Before vs between'] = append_symbols_dict(ape_mav_vn_1, ape_mav_vn_3)
symbols_dict_B['MAV - Whole vs between'] = append_symbols_dict(ape_mav_vn, ape_mav_vn_3)
symbols_dict_B['MAV - Whole vs before'] = append_symbols_dict(ape_mav_vn, ape_mav_vn_1)

symbols_dict_B['SEV - During vs before'] = append_symbols_dict(ape_sev_vn_2, ape_sev_vn_1)
symbols_dict_B['SEV - During vs whole'] = append_symbols_dict(ape_sev_vn_2, ape_sev_vn)
symbols_dict_B['SEV - During vs between'] = append_symbols_dict(ape_sev_vn_2, ape_sev_vn_3)
symbols_dict_B['SEV - Before vs between'] = append_symbols_dict(ape_sev_vn_1, ape_sev_vn_3)
symbols_dict_B['SEV - Whole vs between'] = append_symbols_dict(ape_sev_vn, ape_sev_vn_3)
symbols_dict_B['SEV - Whole vs before'] = append_symbols_dict(ape_sev_vn, ape_sev_vn_1)

p-value:0.978, symbol:n.s., Cohen's d:0.004
p-value:0.089, symbol:n.s., Cohen's d:0.176
p-value:0.020, symbol:*, Cohen's d:0.275
p-value:0.012, symbol:*, Cohen's d:0.299
p-value:0.250, symbol:n.s., Cohen's d:0.111
p-value:0.048, symbol:*, Cohen's d:0.198
p-value:0.000, symbol:***, Cohen's d:0.607
p-value:0.002, symbol:**, Cohen's d:0.368
p-value:0.000, symbol:***, Cohen's d:0.548
p-value:0.683, symbol:n.s., Cohen's d:0.056
p-value:0.058, symbol:n.s., Cohen's d:0.173
p-value:0.060, symbol:n.s., Cohen's d:0.229


In [14]:
symbols_dict_C = {}
symbols_dict_C['MAV - During vs before'] = append_symbols_dict(ape_mav_been_2, ape_mav_been_1)
symbols_dict_C['MAV - During vs whole'] = append_symbols_dict(ape_mav_been_2, ape_mav_been)
symbols_dict_C['MAV - During vs between'] = append_symbols_dict(ape_mav_been_2, ape_mav_been_3)
symbols_dict_C['MAV - Before vs between'] = append_symbols_dict(ape_mav_been_1, ape_mav_been_3)
symbols_dict_C['MAV - Whole vs between'] = append_symbols_dict(ape_mav_been, ape_mav_been_3)
symbols_dict_C['MAV - Whole vs before'] = append_symbols_dict(ape_mav_been, ape_mav_been_1)

symbols_dict_C['SEV - During vs before'] = append_symbols_dict(ape_sev_been_2, ape_sev_been_1)
symbols_dict_C['SEV - During vs whole'] = append_symbols_dict(ape_sev_been_2, ape_sev_been)
symbols_dict_C['SEV - During vs between'] = append_symbols_dict(ape_sev_been_2, ape_sev_been_3)
symbols_dict_C['SEV - Before vs between'] = append_symbols_dict(ape_sev_been_1, ape_sev_been_3)
symbols_dict_C['SEV - Whole vs between'] = append_symbols_dict(ape_sev_been, ape_sev_been_3)
symbols_dict_C['SEV - Whole vs before'] = append_symbols_dict(ape_sev_been, ape_sev_been_1)

p-value:0.000, symbol:***, Cohen's d:0.601
p-value:0.339, symbol:n.s., Cohen's d:0.108
p-value:0.991, symbol:n.s., Cohen's d:0.001
p-value:0.000, symbol:***, Cohen's d:0.576
p-value:0.270, symbol:n.s., Cohen's d:0.103
p-value:0.000, symbol:***, Cohen's d:0.474
p-value:0.000, symbol:***, Cohen's d:1.202
p-value:0.000, symbol:***, Cohen's d:0.696
p-value:0.000, symbol:***, Cohen's d:0.919
p-value:0.109, symbol:n.s., Cohen's d:0.216
p-value:0.039, symbol:*, Cohen's d:0.190
p-value:0.001, symbol:***, Cohen's d:0.409


In [15]:
symbols_dict_D = {}
symbols_dict_D['MAV - During vs before'] = append_symbols_dict(ape_mav_zheng_2, ape_mav_zheng_1)
symbols_dict_D['MAV - During vs whole'] = append_symbols_dict(ape_mav_zheng_2, ape_mav_zheng)
symbols_dict_D['MAV - During vs between'] = append_symbols_dict(ape_mav_zheng_2, ape_mav_zheng_3)
symbols_dict_D['MAV - Before vs between'] = append_symbols_dict(ape_mav_zheng_1, ape_mav_zheng_3)
symbols_dict_D['MAV - Whole vs between'] = append_symbols_dict(ape_mav_zheng, ape_mav_zheng_3)
symbols_dict_D['MAV - Whole vs before'] = append_symbols_dict(ape_mav_zheng, ape_mav_zheng_1)

symbols_dict_D['SEV - During vs before'] = append_symbols_dict(ape_sev_zheng_2, ape_sev_zheng_1)
symbols_dict_D['SEV - During vs whole'] = append_symbols_dict(ape_sev_zheng_2, ape_sev_zheng)
symbols_dict_D['SEV - During vs between'] = append_symbols_dict(ape_sev_zheng_2, ape_sev_zheng_3)
symbols_dict_D['SEV - Before vs between'] = append_symbols_dict(ape_sev_zheng_1, ape_sev_zheng_3)
symbols_dict_D['SEV - Whole vs between'] = append_symbols_dict(ape_sev_zheng, ape_sev_zheng_3)
symbols_dict_D['SEV - Whole vs before'] = append_symbols_dict(ape_sev_zheng, ape_sev_zheng_1)

p-value:0.000, symbol:***, Cohen's d:0.867
p-value:0.306, symbol:n.s., Cohen's d:0.110
p-value:0.602, symbol:n.s., Cohen's d:0.063
p-value:0.000, symbol:***, Cohen's d:0.737
p-value:0.113, symbol:n.s., Cohen's d:0.151
p-value:0.000, symbol:***, Cohen's d:0.631
p-value:0.000, symbol:***, Cohen's d:1.011
p-value:0.000, symbol:***, Cohen's d:0.607
p-value:0.000, symbol:***, Cohen's d:0.841
p-value:0.579, symbol:n.s., Cohen's d:0.073
p-value:0.016, symbol:*, Cohen's d:0.223
p-value:0.006, symbol:**, Cohen's d:0.321


In [ ]:
smoothed_width = 3
black_width = 1
smoothed_width_biblio = 1

with plt.style.context(['science', 'notebook', 'grid']):

    KEY_SIZE = 48
    LABEL_SIZE = 40
    TICK_SIZE = 40
    TITLE_SIZE = 46
    LEGEND_SIZE = 36
    DATES_SIZE = 18
    figsize = (32, 10) #figsize = (32, 10)
    
    plt.rc('axes', labelsize=LABEL_SIZE)
    plt.rc('xtick', labelsize=TICK_SIZE)   
    plt.rc('ytick', labelsize=TICK_SIZE)
    plt.rc('figure', titlesize=TITLE_SIZE)
    plt.rc('legend', fontsize=LEGEND_SIZE)
    plt.rcParams['text.usetex'] = True
    
    fig = plt.figure(figsize=figsize, layout="constrained")
    
    ax_dict = fig.subplot_mosaic(
        """
        ABCD
        """,
        gridspec_kw={'wspace': 0.05, 'hspace':0.075}
    )

    for key in ['A', 'B', 'C', 'D']:
        ax_dict[key].set_ylim(-10, 120)
    
    ################################ ------------------------------ C ------------------------------ ################################ 
    key = 'A'
    mav_apes = [ape_mav, ape_mav_1, ape_mav_2, ape_mav_3]
    sev_apes = [ape_sev, ape_sev_1, ape_sev_2, ape_sev_3]
    plot_boxplots_block(ax_dict, key, mav_apes, sev_apes, params_dict)
    plot_stat_signif_block(ax_dict, key, symbols_dict_A, LABEL_SIZE)

    # Specifying xlabels:
    plt.rcParams['text.usetex'] = True
    ax_dict[key].set_xticks(np.arange(len(box_labels)))
    ax_dict[key].set_xticklabels(box_labels, size=LABEL_SIZE)

    # Specifying ylabels:
    ax_dict[key].set_ylabel(y_label)
    ax_dict[key].set_title('The proposed model', size=TITLE_SIZE)
    ax_dict[key].yaxis.set_major_formatter(digit_y_label)

    ################################ ------------------------------ A ------------------------------ ################################ 
    key = 'B'
    mav_apes = [ape_mav_vn, ape_mav_vn_1, ape_mav_vn_2, ape_mav_vn_3]
    sev_apes = [ape_sev_vn, ape_sev_vn_1, ape_sev_vn_2, ape_sev_vn_3]
    plot_boxplots_block(ax_dict, key, mav_apes, sev_apes, params_dict)
    plot_stat_signif_block(ax_dict, key, symbols_dict_B, LABEL_SIZE)
        
    # Specifying xlabels:
    plt.rcParams['text.usetex'] = True
    ax_dict[key].set_xticks(np.arange(len(box_labels)))
    ax_dict[key].set_xticklabels(box_labels, size=LABEL_SIZE)

    # Specifying ylabels:
    ax_dict[key].set_ylabel(y_label)
    ax_dict[key].set_title('van Nuijs et al., 2011', size=TITLE_SIZE)
    ax_dict[key].yaxis.set_major_formatter(digit_y_label)


    ################################ ------------------------------ B ------------------------------ ################################ 
    key = 'C'
    mav_apes = [ape_mav_been, ape_mav_been_1, ape_mav_been_2, ape_mav_been_3]
    sev_apes = [ape_sev_been, ape_sev_been_1, ape_sev_been_2, ape_sev_been_3]
    plot_boxplots_block(ax_dict, key, mav_apes, sev_apes, params_dict)
    plot_stat_signif_block(ax_dict, key, symbols_dict_C, LABEL_SIZE)
            
    # Specifying xlabels:
    plt.rcParams['text.usetex'] = True
    ax_dict[key].set_xticks(np.arange(len(box_labels)))
    ax_dict[key].set_xticklabels(box_labels, size=LABEL_SIZE)

    # Specifying ylabels:
    ax_dict[key].set_ylabel(y_label)
    ax_dict[key].set_title('Been et al., 2014', size=TITLE_SIZE)
    ax_dict[key].yaxis.set_major_formatter(digit_y_label)


    ################################ ------------------------------ D ------------------------------ ################################ 
    key = 'D'
    mav_apes = [ape_mav_zheng, ape_mav_zheng_1, ape_mav_zheng_2, ape_mav_zheng_3]
    sev_apes = [ape_sev_zheng, ape_sev_zheng_1, ape_sev_zheng_2, ape_sev_zheng_3]
    plot_boxplots_block(ax_dict, key, mav_apes, sev_apes, params_dict)
    plot_stat_signif_block(ax_dict, key, symbols_dict_D, LABEL_SIZE)

    # Specifying xlabels:
    plt.rcParams['text.usetex'] = True
    ax_dict[key].set_xticks(np.arange(len(box_labels)))
    ax_dict[key].set_xticklabels(box_labels, size=LABEL_SIZE)

    # Specifying ylabels:
    ax_dict[key].set_ylabel(y_label)
    ax_dict[key].set_title('Zheng et al., 2019', size=TITLE_SIZE)
    ax_dict[key].yaxis.set_major_formatter(digit_y_label)

    # Display subplot keys
    plt.rcParams['text.usetex'] = False
    fig.canvas.draw()

    # Function to align text with the ylabel of a specific axis
    def align_text_with_ylabel(ax, text, fig):
        ylabel = ax.yaxis.label
        bbox = ylabel.get_window_extent()
        bbox_fig = fig.transFigure.inverted().transform(bbox)
        ylabel_center_fig_x = (bbox_fig[0, 0] + bbox_fig[1, 0]) / 2
        ylabel_center_fig_y = (bbox_fig[0, 1] + bbox_fig[1, 1]) / 2
        fig.text(ylabel_center_fig_x, ylabel_center_fig_y + 0.525, text, ha='center', va='center', size=KEY_SIZE, weight='bold')

    # Align text with the ylabels for each subplot
    for n, (key, ax) in enumerate(ax_dict.items()):
        align_text_with_ylabel(ax, key, fig)

    
    plt.rcParams['text.usetex'] = False
    h1, l1 = ax_dict[key].get_legend_handles_labels()
    h1, l1 = h1[:4], l1[:4]
    fig.legend(h1, l1, loc='upper center', bbox_to_anchor=(0.5, 0), fancybox=True, shadow=True, ncol=2)
    plt.savefig('../../outputs/figs/2026-02-17_Model_specific_APE.pdf', bbox_inches = 'tight')